# Etapa 1 — ETL principal (Pipeline_Integrado extendido)

Carga, limpieza y unificación de la base GRD pública 2019–2024.

Este ETL es el **principal** de la tesis. Además de las 19 columnas tradicionales y clínicas (GRD, diagnósticos CIE-10, procedimientos CIE-9-MC), carga las columnas necesarias para construir las **features agregadas** del perfil funcional del hospital:

- **Demográficas**: `SEXO`, `FECHA_NACIMIENTO` (→ `EDAD`).
- **Mix de ingreso**: `TIPO_INGRESO` (urgencia / programada / obstétrica).
- **Tipo de alta**: `TIPOALTA` (domicilio / fallecido / derivación).
- **Procedencia**: `TIPO_PROCEDENCIA` (emergencia / referencia).
- **Pabellón**: `USOSPABELLON`.
- **Obstétrica/neonatal**: `CONDICIONDEALTANEONATO1`, `PESORN1`.

> La dotación de **camas NO se usa** (decisión metodológica: el perfil se construye solo con información derivada del GRD).

**Salida**: `data/processed/grd_filtrado.parquet` (5.8M egresos × 36 cols, con columna `MODALIDAD ∈ {HOSPITALIZACION, CMA}`).

In [1]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert np.__version__.startswith('1.'), 'NumPy debe ser 1.x (ver requirements.txt)'

# Encoding y separador real (|) por año (verificado en exploración inicial)
YEAR_CONFIGS = {
    2019: {'encoding': 'utf-8',      'sep': '|'},
    2020: {'encoding': 'utf-8',      'sep': '|'},
    2021: {'encoding': 'utf-8',      'sep': '|'},
    2022: {'encoding': 'utf-16',     'sep': '|'},
    2023: {'encoding': 'utf-16',     'sep': '|'},
    2024: {'encoding': 'iso-8859-1', 'sep': '|'},
}
for f in sorted(RAW_DIR.glob('*.txt')):
    print(f'  {f.name}  ({f.stat().st_size/1_048_576:.0f} MB)')

  GRD_PUBLICO_2019.txt  (546 MB)
  GRD_PUBLICO_2020.txt  (379 MB)
  GRD_PUBLICO_2021.txt  (409 MB)
  GRD_PUBLICO_2023.txt  (1026 MB)
  GRD_PUBLICO_2024.txt  (540 MB)
  GRD_PUBLICO_EXTERNO_2022.txt  (916 MB)


## 1.1 Columnas a cargar (tradicionales + clínicas + extendidas)

In [2]:
USECOLS = [
    # Identificación
    'COD_HOSPITAL', 'SERVICIO_SALUD',
    # GRD (complejidad)
    'IR_29301_COD_GRD', 'IR_29301_PESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD',
    # Fechas y modalidad
    'FECHA_INGRESO', 'FECHAALTA', 'TIPO_ACTIVIDAD',
    # Demográficas
    'SEXO', 'FECHA_NACIMIENTO',
    # Mix de ingreso / alta / procedencia / especialidad
    'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'TIPOALTA', 'ESPECIALIDAD_MEDICA',
    # Obstétrica / neonatal
    'CONDICIONDEALTANEONATO1', 'PESORN1',
    # Pabellón
    'USOSPABELLON',
    # Diagnósticos CIE-10 y procedimientos CIE-9-MC (5 + 5)
    'DIAGNOSTICO1', 'DIAGNOSTICO2', 'DIAGNOSTICO3', 'DIAGNOSTICO4', 'DIAGNOSTICO5',
    'PROCEDIMIENTO1', 'PROCEDIMIENTO2', 'PROCEDIMIENTO3', 'PROCEDIMIENTO4', 'PROCEDIMIENTO5',
]
print(f'{len(USECOLS)} columnas objetivo')

28 columnas objetivo


## 1.2 Lectura condicional por año (usecols selectivo)

In [3]:
def load_year(year, raw_dir, config, usecols):
    candidates = list(raw_dir.glob(f'*{year}*.txt'))
    if not candidates:
        print(f'[AVISO] sin archivo para {year}'); return pd.DataFrame()
    fp = candidates[0]
    header = pd.read_csv(fp, sep=config['sep'], encoding=config['encoding'], nrows=0)
    header = header.columns.str.strip().str.upper().str.replace(' ', '_').tolist()
    cols = [c for c in usecols if c in header]
    missing = sorted(set(usecols) - set(cols))
    if missing:
        print(f'  [{year}] cols ausentes (se omiten): {missing}')
    df = pd.read_csv(fp, sep=config['sep'], encoding=config['encoding'],
                     low_memory=False, dtype=str, usecols=cols)
    df.columns = [c.strip().upper().replace(' ', '_') for c in df.columns]
    if year == 2024 and 'ID_BENEFICIARIO' in df.columns:
        df.rename(columns={'ID_BENEFICIARIO': 'CIP_ENCRIPTADO'}, inplace=True)
    df['anio'] = year
    print(f'  {year}: {len(df):,} egresos | {df.shape[1]} cols')
    return df

t0 = time.perf_counter()
dfs = []
for year, cfg in YEAR_CONFIGS.items():
    d = load_year(year, RAW_DIR, cfg, USECOLS)
    if not d.empty:
        dfs.append(d)
grd_raw = pd.concat(dfs, ignore_index=True, sort=False)
del dfs; gc.collect()
print(f'\nConcatenado: {len(grd_raw):,} egresos | {grd_raw.shape[1]} cols | {time.perf_counter()-t0:.0f}s')

  2019: 1,151,475 egresos | 29 cols


  2020: 781,912 egresos | 29 cols


  2021: 816,909 egresos | 29 cols


  2022: 932,840 egresos | 29 cols


  2023: 1,039,587 egresos | 29 cols


  2024: 1,085,813 egresos | 29 cols



Concatenado: 5,808,536 egresos | 29 cols | 28s


## 1.3 Filtro por TIPO_ACTIVIDAD y columna MODALIDAD

In [4]:
TIPOS_HOSP = ['HOSPITALIZACIÓN', 'HOSPITALIZACIÓN EN URGENCIA', 'HOSPITALIZACIÓN DIURNA']
TIPO_CMA = 'CIRUGÍA MAYOR AMBULATORIA (CMA)'
is_hosp = grd_raw['TIPO_ACTIVIDAD'].isin(TIPOS_HOSP)
is_cma = grd_raw['TIPO_ACTIVIDAD'] == TIPO_CMA
mask = is_hosp | is_cma
grd = grd_raw[mask].copy()
grd['MODALIDAD'] = np.where(is_cma[mask], 'CMA', 'HOSPITALIZACION')
del grd_raw; gc.collect()
grd = grd[grd['COD_HOSPITAL'].notna() & (grd['COD_HOSPITAL'].astype(str).str.strip() != '')]
print(f"Filtrado: {len(grd):,} egresos")
print(f"  HOSPITALIZACION: {(grd['MODALIDAD']=='HOSPITALIZACION').sum():,}")
print(f"  CMA:             {(grd['MODALIDAD']=='CMA').sum():,}")

Filtrado: 5,808,515 egresos
  HOSPITALIZACION: 4,918,647
  CMA:             889,868


## 1.4 Derivar fechas, estancia, edad y normalizar categóricas

In [5]:
# Fechas: 2023 viene en DD-MM-YYYY; el resto en YYYY-MM-DD
mask_2023 = grd['anio'] == 2023
for raw_c, dt_c in [('FECHA_INGRESO','FECHA_INGRESO_dt'), ('FECHAALTA','FECHAALTA_dt'),
                    ('FECHA_NACIMIENTO','FECHA_NACIMIENTO_dt')]:
    if raw_c not in grd.columns:
        grd[dt_c] = pd.NaT; continue
    grd[dt_c] = pd.NaT
    if mask_2023.any():
        grd.loc[mask_2023, dt_c] = pd.to_datetime(grd.loc[mask_2023, raw_c],
            format='mixed', dayfirst=True, errors='coerce')
    grd.loc[~mask_2023, dt_c] = pd.to_datetime(grd.loc[~mask_2023, raw_c],
        format='mixed', dayfirst=False, errors='coerce')

grd['DIAS_ESTADA'] = (grd['FECHAALTA_dt'] - grd['FECHA_INGRESO_dt']).dt.days.clip(lower=0)
grd['EDAD'] = ((grd['FECHA_INGRESO_dt'] - grd['FECHA_NACIMIENTO_dt']).dt.days / 365.25).clip(0, 120)

for col in ['IR_29301_PESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD']:
    grd[col] = pd.to_numeric(grd[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
if 'PESORN1' in grd.columns:
    grd['PESORN1'] = pd.to_numeric(grd['PESORN1'].astype(str).str.replace(',', '.', regex=False), errors='coerce')

def norm_str(s):
    return s.astype(str).str.strip().str.upper().str.replace(' ', '_')
for raw_c, n_c in [('TIPO_INGRESO','TIPO_INGRESO_N'), ('TIPOALTA','TIPOALTA_N'),
                   ('TIPO_PROCEDENCIA','TIPO_PROCEDENCIA_N')]:
    if raw_c in grd.columns:
        grd[n_c] = norm_str(grd[raw_c])
if 'USOSPABELLON' in grd.columns:
    grd['USOSPABELLON_N'] = pd.to_numeric(grd['USOSPABELLON'], errors='coerce').fillna(0).astype(int)

print('Estancia (mediana):', grd['DIAS_ESTADA'].median(), '| Edad (mediana):', round(grd['EDAD'].median(),1))

Estancia (mediana): 3.0 | Edad (mediana): 44.4


## 1.5 Persistir `grd_filtrado.parquet`

In [6]:
cols_drop = ['FECHA_INGRESO', 'FECHAALTA', 'FECHA_NACIMIENTO']
grd_persist = grd.drop(columns=[c for c in cols_drop if c in grd.columns])
out = PROCESSED_DIR / 'grd_filtrado.parquet'
grd_persist.to_parquet(out, index=False, compression='zstd')
print(f'Persistido: {out.name} | {grd_persist.shape} | {out.stat().st_size/1e6:.0f} MB')
print('Columnas:', sorted(grd_persist.columns.tolist()))

Persistido: grd_filtrado.parquet | (5808515, 36) | 151 MB
Columnas: ['COD_HOSPITAL', 'CONDICIONDEALTANEONATO1', 'DIAGNOSTICO1', 'DIAGNOSTICO2', 'DIAGNOSTICO3', 'DIAGNOSTICO4', 'DIAGNOSTICO5', 'DIAS_ESTADA', 'EDAD', 'ESPECIALIDAD_MEDICA', 'FECHAALTA_dt', 'FECHA_INGRESO_dt', 'FECHA_NACIMIENTO_dt', 'IR_29301_COD_GRD', 'IR_29301_MORTALIDAD', 'IR_29301_PESO', 'IR_29301_SEVERIDAD', 'MODALIDAD', 'PESORN1', 'PROCEDIMIENTO1', 'PROCEDIMIENTO2', 'PROCEDIMIENTO3', 'PROCEDIMIENTO4', 'PROCEDIMIENTO5', 'SERVICIO_SALUD', 'SEXO', 'TIPOALTA', 'TIPOALTA_N', 'TIPO_ACTIVIDAD', 'TIPO_INGRESO', 'TIPO_INGRESO_N', 'TIPO_PROCEDENCIA', 'TIPO_PROCEDENCIA_N', 'USOSPABELLON', 'USOSPABELLON_N', 'anio']


## 1.6 Resumen Etapa 1

El `grd_filtrado.parquet` resultante es la **única fuente canónica** para las etapas siguientes. Contiene tanto Hospitalización como CMA (vía `MODALIDAD`) y todas las variables necesarias para las features agregadas.

In [7]:
print('=' * 60)
print('RESUMEN ETAPA 1 — ETL principal')
print('=' * 60)
print(f'Egresos totales:   {len(grd_persist):,}')
print(f'Hospitalización:   {(grd_persist["MODALIDAD"]=="HOSPITALIZACION").sum():,}')
print(f'CMA:               {(grd_persist["MODALIDAD"]=="CMA").sum():,}')
print(f'Hospitales (raw):  {grd_persist["COD_HOSPITAL"].nunique()}')
print(f'Años:              {sorted(grd_persist["anio"].unique())}')
print('\nSiguiente: 02_eda.ipynb')

RESUMEN ETAPA 1 — ETL principal
Egresos totales:   5,808,515
Hospitalización:   4,918,647


CMA:               889,868


Hospitales (raw):  72
Años:              [2019, 2020, 2021, 2022, 2023, 2024]

Siguiente: 02_eda.ipynb
